# Boat model explorer

A working model of the boat from `optimization2.md`, for poking at. Change an input, re-run, and see what
moves. Units are imperial (ft, in, lb, mph, Wh) because that is how the design is discussed.

**What is real and what is not**

| Piece | Status |
| --- | --- |
| Planing equilibrium (trim, wetted length, thrust needed) | Real: [OpenPlaning](https://github.com/elcf/python-openplaning), Savitsky's empirical method. Prismatic hull only. |
| Hydrostatics, freeboard, hull mass | Ours, closed form for a prismatic V-hull. The **prismatic coefficient is a guess**. |
| Propeller rpm, tip speed, Kv | Your equation: `rpm = mph x 1056 / (pitch x 0.85)`, `Kv = rpm / loaded voltage`. |
| Battery voltage under load | Ours: `V = (Voc + sqrt(Voc^2 - 4 R P)) / 2`. |
| Cell data, motor mass fit, propeller efficiency, current limit | **Placeholders**. Every one is marked `PLACEHOLDER` below. |

Nothing here is validated against a real boat. Use it to learn which variables matter and where the
constraints bite, not to read off a design.

**Run it:** `optimization/.venv/bin/jupyter lab optimization/explore.ipynb`, then Run All. The optimizer
cell near the end takes about a minute.

## 1. Setup

In [ ]:
import math, time, warnings
from dataclasses import dataclass, field, replace

warnings.filterwarnings("ignore", message="pkg_resources")   # OpenPlaning imports it; harmless

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from openplaning import PlaningBoat

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})

# unit conversions to SI (OpenPlaning works in SI)
LB_N, IN, FT, MPH = 4.44822162, 0.0254, 0.3048, 0.44704
LB_FT3_TO_KG_M3 = 16.01846

## 2. Inputs

Two groups. `Design` holds the decision variables the optimizer would choose. `Fixed` holds everything held
constant: rules, environment, and placeholder assumptions. Edit the defaults here or override them in later
cells with `replace(design, beam_in=20)`.

In [ ]:
@dataclass
class Cell:
    # PLACEHOLDER numbers for a generic high-drain 21700 cell. Not from a datasheet.
    ah: float = 5.0
    v_nominal: float = 3.6
    v_full: float = 4.2
    resistance_ohm: float = 0.015
    mass_lb: float = 0.154
    max_current_a: float = 15.0          # continuous, per cell
    soc: tuple = (0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9, 1.0)
    ocv: tuple = (3.00, 3.45, 3.55, 3.65, 3.75, 3.92, 4.00, 4.10, 4.20)   # generic NMC curve
    pack_overhead: float = 0.15          # case, BMS, wiring as a fraction of cell mass


@dataclass
class Fixed:
    # rules
    payload_lb: float = 30.0
    freeboard_rule_in: float = 3.0
    voltage_limit_v: float = 55.5
    # environment (wave height and current are TO BE SET)
    water_lb_ft3: float = 62.4                # fresh
    water_nu: float = 1.14e-6                 # kinematic viscosity, m^2/s
    wave_height_in: float = 4.0
    wave_allowance_factor: float = 0.5        # freeboard must cover H/2 on top of the rule
    current_mph: float = 1.0
    course_mode: str = "worst"                # "worst": whole course against the current; "outback": equal legs
    # mission
    course_miles: float = 2.0
    allowance_s: float = 20.0
    hotel_w: float = 30.0
    # design margins
    usable_energy_fraction: float = 0.70      # of the energy in the pack at the start of the race
    rating_fraction: float = 0.80
    lcg_band: tuple = (0.30, 0.45)            # fraction of length from the transom
    min_cell_voltage_end: float = 3.0
    # hull construction (synthetic)
    laminate_lb_ft2: float = 0.41
    hull_structure_lb: float = 4.4
    fixed_equipment_lb: float = 8.8
    prismatic_coeff: float = 0.65             # PLACEHOLDER: a pure prism overstates static volume
    # heights above the keel (in) and radius of gyration for the mass model; all PLACEHOLDER
    payload_z_in: float = 8.0
    battery_z_in: float = 4.0
    motor_z_in: float = 4.0
    fixed_z_in: float = 8.0
    hull_z_frac_of_depth: float = 0.45
    gyradius_frac: float = 0.25               # of length
    thrust_height_in: float = 4.0
    # propulsion (PLACEHOLDER)
    prop_efficiency: float = 0.65             # no propeller model yet, so pitch and diameter do not change this
    prop_mass_lb: float = 0.5
    motor_max_voltage_v: float = 60.0
    slip_factor: float = 0.85                 # your rpm equation
    cell: Cell = field(default_factory=Cell)


@dataclass
class Design:
    # hull (prismatic)
    length_ft: float = 8.5
    beam_in: float = 16.0
    deadrise_deg: float = 10.0
    depth_in: float = 12.0
    # propulsion
    motor_kw: float = 4.0
    motor_eff: float = 0.90
    pitch_in: float = 10.0
    prop_diam_in: float = 6.0
    thrust_angle_deg: float = 0.0
    # battery
    series: int = 14
    parallel: int = 6
    # placement (fraction of length forward of the transom)
    payload_x: float = 0.30
    battery_x: float = 0.35
    # operation
    speed_mph: float = 22.0


FIXED, DESIGN = Fixed(), Design()

## 3. Hull: prismatic hydrostatics and mass

V-bottom section of chine beam `b` and deadrise `beta`, vertical sides up to the hull depth, the same from transom to
bow. The chine sits `b/2 * tan(beta)` above the keel. Static volume is `prismatic coefficient x section area x
length`. Freeboard is measured at rest, which is conservative because the boat rides higher when planing.

In [ ]:
def chine_height_ft(d):
    return d.beam_in / 12 / 2 * math.tan(math.radians(d.deadrise_deg))


def section_area_ft2(draft_ft, d):
    b, t = d.beam_in / 12, math.tan(math.radians(d.deadrise_deg))
    hc = b / 2 * t
    return draft_ft ** 2 / t if draft_ft <= hc else b * hc / 2 + b * (draft_ft - hc)


def static_draft_ft(mass_lb, d, f):
    # invert section_area_ft2 for the draft that floats the boat
    area = (mass_lb / f.water_lb_ft3) / (f.prismatic_coeff * d.length_ft)
    b, t = d.beam_in / 12, math.tan(math.radians(d.deadrise_deg))
    hc = b / 2 * t
    a_chine = b * hc / 2
    return math.sqrt(area * t) if area <= a_chine else hc + (area - a_chine) / b


def shell_area_ft2(d):
    b, L, D = d.beam_in / 12, d.length_ft, d.depth_in / 12
    hc = chine_height_ft(d)
    side_h = max(D - hc, 0.0)
    bottom = b * L / math.cos(math.radians(d.deadrise_deg))
    return bottom + 2 * side_h * L + b * L + b * side_h + b * hc / 2   # bottom, sides, deck, transom


def hull_mass_lb(d, f):
    return shell_area_ft2(d) * f.laminate_lb_ft2 + f.hull_structure_lb

## 4. Battery

`capacity = S x P x cell Ah x cell nominal volts`. The start-of-race charge is **derived**: the highest charge at which the
open-circuit voltage stays at or under the 55.5 V rule (capped at 100%). Under load the voltage sags by the pack's internal
resistance: solving `V = Voc - I R` with `I = P / V` gives `V = (Voc + sqrt(Voc^2 - 4 R P)) / 2`, which only has a solution
while `Voc^2 >= 4 R P`.

In [ ]:
def ocv(soc, cell):
    return float(np.interp(soc, cell.soc, cell.ocv))


def start_soc(series, f):
    cell = f.cell
    if series * cell.v_full <= f.voltage_limit_v:
        return 1.0
    return float(np.interp(f.voltage_limit_v / series, cell.ocv, cell.soc))


def pack(d, f):
    c = f.cell
    n = d.series * d.parallel
    return dict(cells=n, capacity_wh=n * c.ah * c.v_nominal,
                mass_lb=n * c.mass_lb * (1 + c.pack_overhead),
                resistance_ohm=d.series / d.parallel * c.resistance_ohm,
                current_limit_a=d.parallel * c.max_current_a)


def loaded_voltage(v_oc, r_pack, p_bus):
    disc = v_oc ** 2 - 4 * r_pack * p_bus
    return (v_oc + math.sqrt(disc)) / 2 if disc >= 0 else float("nan")

## 5. Motor and propeller

Motor mass is a **placeholder fit**: it rises with rated power and with peak efficiency, so the optimizer cannot simply take the
biggest, most efficient motor. Replace it with a fit to real products.

In [ ]:
def motor_mass_lb(d):
    # PLACEHOLDER: 1 lb + 1.6 lb/kW, +6% per point of efficiency above 90%
    return (1.0 + 1.6 * d.motor_kw) * (1 + 6 * (d.motor_eff - 0.90))


def prop_numbers(d, f, speed_mph, v_loaded):
    rpm = speed_mph * 1056 / (d.pitch_in * f.slip_factor)
    tip_fps = math.pi * d.prop_diam_in / 12 * rpm / 60
    return dict(rpm=rpm, tip_speed_fps=tip_fps, kv_needed=rpm / v_loaded if v_loaded == v_loaded else float("nan"))

## 6. Planing equilibrium (OpenPlaning)

For a given weight, LCG, VCG, speed, beam and deadrise, OpenPlaning finds the trim and wetted length where lift balances
weight and moments balance, and returns the thrust the propeller must supply. If it cannot find a solution the design is
treated as infeasible.

We also record Savitsky's applicability limits so we can enforce them (or not; see `ENFORCE_VALIDITY` below):
running trim 2 to 15 degrees, mean wetted length over beam at most 4, speed coefficient `Cv` 0.6 to 13, and the wetted
keel no longer than the hull.

In [ ]:
def planing(d, f, mass_lb, lcg_ft, vcg_m, speed_mph):
    # OpenPlaning warns on every call outside a method's range, and turns warnings back on internally
    # after each solve, so silence them for the duration of the call. We check the limits ourselves.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return _planing(d, f, mass_lb, lcg_ft, vcg_m, speed_mph)


def _planing(d, f, mass_lb, lcg_ft, vcg_m, speed_mph):
    try:
        b = PlaningBoat(speed=speed_mph * MPH, weight=mass_lb * LB_N, beam=d.beam_in * IN, lcg=lcg_ft * FT, vcg=vcg_m,
                        r_g=f.gyradius_frac * d.length_ft * FT, beta=d.deadrise_deg, epsilon=d.thrust_angle_deg,
                        vT=f.thrust_height_in * IN, lT=0.0, loa=d.length_ft * FT, H_sig=f.wave_height_in * IN,
                        rho=f.water_lb_ft3 * LB_FT3_TO_KG_M3, nu=f.water_nu)
        b.get_steady_trim()
        warnings.simplefilter("ignore")        # get_steady_trim re-enables warnings when it finishes
        b.get_forces()
        thrust = abs(float(b.thrust_force[0]))
        out = dict(ok=bool(np.isfinite(thrust)), trim_deg=float(b.tau), thrust_n=thrust, keel_m=float(b.L_K),
                   chine_m=float(b.L_C), lam=float(b.lambda_W), transom_draft_m=float(b.T))
        out["cv"] = speed_mph * MPH / math.sqrt(9.80665 * d.beam_in * IN)
        return out, b
    except Exception as e:                      # OpenPlaning raises when no equilibrium exists
        return dict(ok=False, error=str(e).splitlines()[0][:80]), None

## 7. Whole-boat evaluation

`evaluate(design, fixed)` walks the chain: hull, mass and balance, hydrostatics, planing thrust, propeller, power, battery,
race time, and then every constraint. Each constraint is a **margin** (positive passes) with a scale so violations can be
compared, grouped as `rule`, `limit`, `geometry` or `validity`.

Race time uses speed over the ground: `speed - current`. The course allowance is charged at cruise power.

In [ ]:
ENFORCE_VALIDITY = True     # treat Savitsky's applicability limits as hard constraints


@dataclass
class Result:
    m: dict                 # metrics
    margins: dict           # name -> (margin, scale, group)

    def feasible(self, enforce_validity=None):
        ev = ENFORCE_VALIDITY if enforce_validity is None else enforce_validity
        return all(v >= 0 for v, s, g in self.margins.values() if ev or g != "validity")

    def violation(self, enforce_validity=None):
        ev = ENFORCE_VALIDITY if enforce_validity is None else enforce_validity
        return sum(max(0.0, -v / s) for v, s, g in self.margins.values() if ev or g != "validity")

    def failing(self, enforce_validity=None):
        ev = ENFORCE_VALIDITY if enforce_validity is None else enforce_validity
        return [k for k, (v, s, g) in self.margins.items() if v < 0 and (ev or g != "validity")]


def evaluate(d, f=FIXED, speed_mph=None):
    v = d.speed_mph if speed_mph is None else speed_mph
    cell = f.cell
    m, g = {}, {}

    # hull, masses, balance
    hull_lb, mot_lb, bat = hull_mass_lb(d, f), motor_mass_lb(d), pack(d, f)
    parts = [  # name, lb, x as fraction of length, z in above keel
        ("hull", hull_lb, 0.45, f.hull_z_frac_of_depth * d.depth_in),
        ("payload", f.payload_lb, d.payload_x, f.payload_z_in),
        ("battery", bat["mass_lb"], d.battery_x, f.battery_z_in),
        ("motor", mot_lb, 0.10, f.motor_z_in),
        ("prop", f.prop_mass_lb, 0.0, f.motor_z_in),
        ("fixed", f.fixed_equipment_lb, 0.40, f.fixed_z_in),
    ]
    mass = sum(p[1] for p in parts)
    lcg_ft = sum(p[1] * p[2] for p in parts) / mass * d.length_ft
    vcg_m = sum(p[1] * p[3] for p in parts) / mass * IN
    m.update(mass_lb=mass, hull_lb=hull_lb, motor_lb=mot_lb, battery_lb=bat["mass_lb"],
             lcg_frac=lcg_ft / d.length_ft, capacity_wh=bat["capacity_wh"])

    # hydrostatics
    draft_in = static_draft_ft(mass, d, f) * 12
    wave_in = f.wave_allowance_factor * f.wave_height_in
    m.update(static_draft_in=draft_in, freeboard_in=d.depth_in - draft_in)
    g["freeboard"] = (d.depth_in - draft_in - f.freeboard_rule_in - wave_in, 3.0, "rule")
    g["depth_over_chine"] = (d.depth_in - chine_height_ft(d) * 12, 12.0, "geometry")

    # planing thrust at this speed through the water
    p, boat = planing(d, f, mass, lcg_ft, vcg_m, v)
    m["planing"] = p
    ground = v - f.current_mph
    g["ground_speed"] = (ground, 5.0, "rule")
    if f.course_mode == "worst":
        hours = f.course_miles / ground if ground > 0 else float("inf")
    else:
        hours = (f.course_miles / 2 / (v - f.current_mph) + f.course_miles / 2 / (v + f.current_mph)) if ground > 0 else float("inf")
    race_s = hours * 3600 + f.allowance_s
    m["race_time_s"] = race_s

    g["solver_ok"] = (1.0 if p["ok"] else -1.0, 1.0, "validity")
    if not p["ok"]:
        return Result(m, g)

    # validity limits of the planing method
    g["trim_min"] = (p["trim_deg"] - 2.0, 2.0, "validity")
    g["trim_max"] = (15.0 - p["trim_deg"], 15.0, "validity")
    g["wetted_length_ratio"] = (4.0 - p["lam"], 4.0, "validity")
    g["cv_min"] = (p["cv"] - 0.6, 0.6, "validity")
    g["cv_max"] = (13.0 - p["cv"], 13.0, "validity")
    g["keel_within_hull"] = (d.length_ft * FT - p["keel_m"], d.length_ft * FT, "validity")

    # power chain
    shaft_w = p["thrust_n"] * v * MPH / f.prop_efficiency
    bus_w = shaft_w / d.motor_eff + f.hotel_w
    energy_wh = bus_w * race_s / 3600
    m.update(thrust_n=p["thrust_n"], trim_deg=p["trim_deg"], shaft_w=shaft_w, bus_w=bus_w, energy_used_wh=energy_wh)

    # battery and voltage
    soc0 = start_soc(d.series, f)
    v_oc = d.series * ocv(soc0, cell)
    v_load = loaded_voltage(v_oc, bat["resistance_ohm"], bus_w)
    current = bus_w / v_load if v_load == v_load else float("inf")
    soc_end = soc0 - energy_wh / bat["capacity_wh"]
    v_end = loaded_voltage(d.series * ocv(max(soc_end, 0.0), cell), bat["resistance_ohm"], bus_w)
    m.update(start_soc=soc0, v_oc_start=v_oc, v_loaded=v_load, current_a=current, soc_end=soc_end,
             v_loaded_end=v_end, **prop_numbers(d, f, v, v_load))

    # limits
    motor_i_rating = d.motor_kw * 1000 / (d.motor_eff * d.series * cell.v_nominal)
    usable_wh = f.usable_energy_fraction * bat["capacity_wh"] * soc0
    g["voltage_rule"] = (f.voltage_limit_v - v_oc, f.voltage_limit_v, "rule")
    g["motor_max_voltage"] = (f.motor_max_voltage_v - d.series * cell.v_full, f.motor_max_voltage_v, "limit")
    g["energy"] = (usable_wh - energy_wh, max(usable_wh, 1.0), "limit")
    g["shaft_power"] = (f.rating_fraction * d.motor_kw * 1000 - shaft_w, d.motor_kw * 1000, "limit")
    g["battery_current"] = (f.rating_fraction * bat["current_limit_a"] - current, bat["current_limit_a"], "limit")
    g["motor_current"] = (f.rating_fraction * motor_i_rating - current, motor_i_rating, "limit")
    g["pack_can_deliver"] = (1.0 if v_load == v_load else -1.0, 1.0, "limit")
    g["end_cell_voltage"] = ((v_end / d.series - f.min_cell_voltage_end) if v_end == v_end else -1.0, 1.0, "limit")
    g["lcg_min"] = (m["lcg_frac"] - f.lcg_band[0], 0.1, "limit")
    g["lcg_max"] = (f.lcg_band[1] - m["lcg_frac"], 0.1, "limit")
    return Result(m, g)


def show(res, enforce_validity=None, title=None):
    m, p = res.m, res.m["planing"]
    if title: print(title)
    line = lambda k, v, u="": print(f"  {k:<26}{v:>10}  {u}")
    line("total mass", f"{m['mass_lb']:.1f}", f"lb   (hull {m['hull_lb']:.1f}, battery {m['battery_lb']:.1f}, motor {m['motor_lb']:.1f})")
    line("LCG", f"{100 * m['lcg_frac']:.1f}", "% of length from transom")
    line("static draft / freeboard", f"{m['static_draft_in']:.1f} / {m['freeboard_in']:.1f}", "in")
    if p["ok"]:
        line("running trim", f"{m['trim_deg']:.2f}", f"deg   wetted length/beam {p['lam']:.2f}, Cv {p['cv']:.2f}")
        line("thrust needed", f"{m['thrust_n']:.0f}", "N")
        line("shaft / bus power", f"{m['shaft_w'] / 1000:.2f} / {m['bus_w'] / 1000:.2f}", "kW")
        line("pack", f"{m['capacity_wh']:.0f}", f"Wh; start charge {100 * m['start_soc']:.0f}%, Voc {m['v_oc_start']:.1f} V, loaded {m['v_loaded']:.1f} V, {m['current_a']:.0f} A")
        line("race energy", f"{m['energy_used_wh']:.0f}", "Wh")
        line("propeller", f"{m['rpm']:.0f}", f"rpm, tip {m['tip_speed_fps']:.0f} ft/s, Kv needed {m['kv_needed']:.0f}")
    else:
        line("planing solver", "FAILED", p.get("error", ""))
    t = m["race_time_s"]
    line("race time", f"{int(t // 60)}:{t % 60:04.1f}", "min:sec")
    bad = res.failing(enforce_validity)
    print("  " + ("FEASIBLE" if not bad else "FAILS: " + ", ".join(bad)))
    return res

## 8. The fastest feasible speed

Speed is not really a free choice: for any design there is a highest speed at which every constraint still holds, and that is
the speed the boat should run. `fastest_feasible` scans speeds, keeps the feasible ones, and refines the upper edge. Race time
follows from it, so the optimizer only has to search the other variables.

In [ ]:
def fastest_feasible(d, f=FIXED, lo=8.0, hi=60.0, step=4.0, tol=0.1, enforce_validity=None):
    ok = lambda s: evaluate(replace(d, speed_mph=s), f, s)
    feas = lambda r: r.feasible(enforce_validity)
    best = None
    for s in np.arange(lo, hi + 1e-9, step):
        r = ok(s)
        if feas(r): best = (s, r)
    if best is None:
        return None
    s_lo, r_lo = best
    s_hi = s_lo + step
    if s_hi <= hi:
        while s_hi - s_lo > tol:
            mid = (s_lo + s_hi) / 2
            r = ok(mid)
            if feas(r): s_lo, r_lo = mid, r
            else: s_hi = mid
    return s_lo, r_lo


def limiting(d, f=FIXED, nudge=0.5):
    # which constraints fail just above the fastest feasible speed: what is stopping the boat
    b = fastest_feasible(d, f)
    if b is None:
        return None, []
    v, _ = b
    return v, evaluate(replace(d, speed_mph=v + nudge), f, v + nudge).failing()

## 9. Baseline design

In [ ]:
res = show(evaluate(DESIGN, FIXED), title=f"Baseline at {DESIGN.speed_mph:.0f} mph")

In [ ]:
best = fastest_feasible(DESIGN, FIXED)
if best:
    v, r = best
    show(r, title=f"Fastest feasible speed for the baseline design: {v:.1f} mph")
    print("  Just above that speed these constraints fail:", limiting(DESIGN, FIXED)[1])
else:
    print("No feasible speed. Failing at 22 mph:", evaluate(DESIGN, FIXED).failing())

## 10. Which constraint stops the boat?

At each speed, the margin of every constraint as a fraction of its scale. The first line to cross zero is the one limiting speed.

In [ ]:
speeds = np.arange(8, 41, 1.0)
rows = [evaluate(replace(DESIGN, speed_mph=s), FIXED, s) for s in speeds]
names, groups = [], {}
for r in rows:
    for k, (_, _, grp) in r.margins.items():
        if k not in groups: names.append(k); groups[k] = grp

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
for k in names:
    if groups[k] == "geometry": continue
    y = [r.margins[k][0] / r.margins[k][1] if k in r.margins else np.nan for r in rows]
    ax.plot(speeds, y, label=k, lw=1.4, ls="--" if groups[k] == "validity" else "-")
ax.axhline(0, color="k", lw=1)
ax.set_ylim(-1, 1.5); ax.set_xlabel("speed (mph)"); ax.set_ylabel("margin / scale (below 0 fails)")
ax.set_title("Constraint margins vs speed (dashed = planing validity)"); ax.legend(fontsize=6, ncol=2)

ax = axes[1]
ok = np.array([r.m["planing"]["ok"] for r in rows])
thr = np.array([r.m.get("thrust_n", np.nan) for r in rows])
bus = np.array([r.m.get("bus_w", np.nan) for r in rows]) / 1000
ax.plot(speeds, thr, color="C0"); ax.set_ylabel("thrust needed (N)", color="C0"); ax.set_xlabel("speed (mph)")
ax2 = ax.twinx(); ax2.plot(speeds, bus, color="C3"); ax2.set_ylabel("bus power (kW)", color="C3"); ax2.grid(False)
ax.set_title("Thrust and electrical power")
plt.tight_layout(); plt.show()

## 11. Sweep one variable

`sweep` changes one design variable and reports, for each value, the fastest feasible speed and the race time that speed gives.
Gaps mean no speed in the scan was feasible. This is the quickest way to see whether a variable runs to an edge and what stops it.

In [ ]:
def sweep(var, values, f=FIXED, d=DESIGN):
    out = []
    for x in values:
        dd = replace(d, **{var: type(getattr(d, var))(x)})
        b = fastest_feasible(dd, f)
        out.append((x, None if b is None else b[0], None if b is None else b[1].m["race_time_s"],
                    None if b is None else b[1].m["thrust_n"]))
    return out


def plot_sweeps(specs, f=FIXED, d=DESIGN, ncols=3):
    n = len(specs); nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.3 * nrows), squeeze=False)
    for ax, (var, values, label) in zip(axes.flat, specs):
        res = sweep(var, values, f, d)
        xs = [r[0] for r in res]; vs = [np.nan if r[1] is None else r[1] for r in res]
        ax.plot(xs, vs, "o-", ms=3)
        ax.axvline(getattr(d, var), color="gray", ls=":", lw=1)
        ax.set_xlabel(label); ax.set_ylabel("fastest feasible speed (mph)")
        if all(np.isnan(vs)): ax.text(0.5, 0.5, "never feasible", transform=ax.transAxes, ha="center")
    for ax in axes.flat[n:]: ax.axis("off")
    plt.tight_layout(); plt.show()


plot_sweeps([
    ("beam_in", np.arange(10, 40, 1.5), "chine beam (in)"),
    ("deadrise_deg", np.arange(4, 26, 1.5), "deadrise (deg)"),
    ("length_ft", np.arange(6.5, 11.6, 0.5), "length (ft)"),
    ("depth_in", np.arange(6, 20, 1.0), "hull depth (in)"),
    ("payload_x", np.arange(0.20, 0.61, 0.025), "payload position (fraction of length)"),
    ("battery_x", np.arange(0.20, 0.61, 0.025), "battery position (fraction of length)"),
    ("motor_kw", np.arange(1.0, 10.1, 0.5), "motor rated power (kW)"),
    ("motor_eff", np.arange(0.80, 0.971, 0.01), "motor peak efficiency"),
    ("parallel", np.arange(1, 13, 1), "parallel strings"),
])

## 12. Pack size: series and parallel counts

The battery decision variables. Fastest feasible speed for every combination. More series cells raise voltage and cut current; more
parallel strings add capacity and current headroom but also mass. 14S only works because the pack is charged partway to stay under 55.5 V.

In [ ]:
S_vals, P_vals = [12, 13, 14], list(range(1, 13))
grid = np.full((len(S_vals), len(P_vals)), np.nan)
for i, S in enumerate(S_vals):
    for j, P in enumerate(P_vals):
        b = fastest_feasible(replace(DESIGN, series=S, parallel=P), FIXED)
        if b: grid[i, j] = b[0]

fig, ax = plt.subplots(figsize=(8, 2.8))
im = ax.imshow(grid, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(P_vals)), P_vals); ax.set_yticks(range(len(S_vals)), [f"{s}S" for s in S_vals])
ax.set_xlabel("parallel strings"); ax.set_title("Fastest feasible speed (mph); blank = never feasible"); ax.grid(False)
for i in range(len(S_vals)):
    for j in range(len(P_vals)):
        if not np.isnan(grid[i, j]): ax.text(j, i, f"{grid[i, j]:.0f}", ha="center", va="center", color="w", fontsize=8)
plt.colorbar(im, ax=ax, label="mph"); plt.tight_layout(); plt.show()

## 13. Interactive

Drag the sliders to change the design and see the full report at the current speed. (Needs `ipywidgets`; skipped silently if it is
missing.)

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import clear_output

    fields = [
        ("length_ft", 7.0, 10.5, 0.25), ("beam_in", 10, 40, 0.5), ("deadrise_deg", 4, 25, 0.5), ("depth_in", 6, 20, 0.5),
        ("payload_x", 0.2, 0.6, 0.01), ("battery_x", 0.2, 0.6, 0.01), ("motor_kw", 1, 10, 0.25),
        ("motor_eff", 0.8, 0.97, 0.005), ("series", 12, 14, 1), ("parallel", 1, 12, 1), ("speed_mph", 8, 40, 0.5),
        ("pitch_in", 3, 24, 0.5), ("prop_diam_in", 3, 10, 0.5),
    ]
    sliders = {n: (w.IntSlider if isinstance(getattr(DESIGN, n), int) else w.FloatSlider)(
        value=getattr(DESIGN, n), min=lo, max=hi, step=st, description=n, continuous_update=False,
        style={"description_width": "110px"}, layout=w.Layout(width="420px")) for n, lo, hi, st in fields}
    enforce = w.Checkbox(value=True, description="enforce Savitsky validity limits")
    out = w.Output()

    def update(_=None):
        d = replace(DESIGN, **{n: s.value for n, s in sliders.items()})
        with out:
            clear_output(wait=True)
            show(evaluate(d, FIXED), enforce_validity=enforce.value, title=f"speed {d.speed_mph} mph")

    for s in list(sliders.values()) + [enforce]: s.observe(update, names="value")
    display(w.HBox([w.VBox(list(sliders.values()) + [enforce]), out]))
    update()
except ImportError:
    print("ipywidgets not installed; skipping the interactive panel")

## 14. Search for a good design

Differential evolution over the design variables, with the fastest feasible speed found for each candidate (so speed is not a
variable). Series and parallel counts are integers. Pitch, diameter and thrust angle are left out because the model has no propeller
efficiency yet, so they cannot change the answer.

This is a global search but not a proof: run it a few times with different seeds and compare. It takes about a minute.
Edit `BOUNDS` to change the search space. The hull length range is the plan's; the beam range is wide on purpose so the Savitsky
validity window shows up by itself.

In [ ]:
BOUNDS = dict(
    length_ft=(7.5, 9.0), beam_in=(10, 36), deadrise_deg=(6, 16), depth_in=(8, 16),
    payload_x=(0.25, 0.50), battery_x=(0.25, 0.45),
    motor_kw=(2.0, 10.0), motor_eff=(0.85, 0.95),
    series=(12, 14), parallel=(1, 12),
)
INTEGER = {"series", "parallel"}
NAMES = list(BOUNDS)


def design_from(x):
    kw = {n: (int(round(v)) if n in INTEGER else float(v)) for n, v in zip(NAMES, x)}
    return replace(DESIGN, **kw)


def objective(x):
    d = design_from(x)
    b = fastest_feasible(d, FIXED, step=6.0, tol=0.25)
    if b is not None:
        return b[1].m["race_time_s"]
    return 1000.0 + evaluate(replace(d, speed_mph=20.0), FIXED, 20.0).violation()   # steer toward feasibility


def run_search(seed=1, popsize=8, maxiter=25):
    t0 = time.time()
    r = differential_evolution(objective, [BOUNDS[n] for n in NAMES], integrality=[n in INTEGER for n in NAMES],
                               seed=seed, popsize=popsize, maxiter=maxiter, tol=1e-6, polish=False, updating="deferred")
    return r, time.time() - t0

In [ ]:
sr, secs = run_search(seed=1)
best_design = design_from(sr.x)
print(f"search finished in {secs:.0f} s, {sr.nfev} candidate designs, best race time {sr.fun:.1f} s")
b = fastest_feasible(best_design, FIXED)
if b:
    best_design = replace(best_design, speed_mph=b[0])     # exact, so it sits right on the limit
    show(evaluate(best_design, FIXED), title=f"Best design found, running at {best_design.speed_mph:.1f} mph")
    print("  Just above that speed these constraints fail:", limiting(best_design, FIXED)[1])
else:
    print("No feasible design was found. Try more iterations or wider bounds.")

In [ ]:
print(f"{'variable':<14}{'value':>10}    position in range")
for n in NAMES:
    lo, hi = BOUNDS[n]; v = getattr(best_design, n); frac = (v - lo) / (hi - lo) if hi > lo else 0.5
    edge = "  <-- at the lower bound" if frac < 0.02 else "  <-- at the upper bound" if frac > 0.98 else ""
    print(f"{n:<14}{v:>10.3f}    {100 * frac:5.0f}%{edge}")

### Reading the result

In a typical run the optimizer takes the narrowest beam and a high deadrise, and it stops on the Savitsky trim floor
(2 degrees), not on motor power. Both are the model telling you what it is missing, not design advice:

- Nothing needs width. There is no roll stability or payload footprint constraint, so beam runs to its lower bound.
- Higher deadrise and narrower beam keep trim above 2 degrees as speed rises, so the search leans on them.
- Speeds far above the plan's 18 to 30 mph target come from an unvalidated resistance estimate. Treat them as a statement about
  the model.

Look at the "at the lower/upper bound" flags: each one is a place where a real constraint (stability, payload, ride) is absent.

## 15. Sanity checks on the regime

Planing needs the boat to be fast enough for dynamic lift. Two dimensionless numbers, computed for the baseline:
the volumetric Froude number `V / sqrt(g x volume^(1/3))` (planing is roughly 2.3 or more) and the beam speed coefficient
`Cv = V / sqrt(g x beam)` (Savitsky's method covers 0.6 to 13). These are rules of thumb, not limits from the method's authors.

In [ ]:
m0 = evaluate(DESIGN, FIXED).m
vol_ft3 = m0["mass_lb"] / FIXED.water_lb_ft3
print(f"loaded mass {m0['mass_lb']:.1f} lb, displaced volume {vol_ft3:.2f} ft^3")
for mph in (8, 10, 12, 15, 18, 22, 26, 30):
    v = mph * 5280 / 3600
    fn = v / math.sqrt(32.174 * vol_ft3 ** (1 / 3))
    cv = v / math.sqrt(32.174 * DESIGN.beam_in / 12)
    print(f"  {mph:>2} mph   volumetric Froude {fn:4.1f}   Cv {cv:4.1f}")

## 16. Try the plan's boat

The plan's dimensions (34 in beam, 8.5 ft, 95 lb class) with the validity limits switched off, to see what OpenPlaning reports when
the boat is much lighter than its beam calls for. Trim falls below 2 degrees, so the result is an extrapolation.

In [ ]:
plan_boat = replace(DESIGN, beam_in=34.0, parallel=4)
for mph in (10, 14, 18, 22, 30):
    r = evaluate(replace(plan_boat, speed_mph=mph), FIXED, mph)
    p = r.m["planing"]
    if p["ok"]:
        print(f"{mph:>3} mph  trim {p['trim_deg']:.2f} deg  thrust {p['thrust_n']:.0f} N  wetted length/beam {p['lam']:.2f}  "
              f"validity failures: {[k for k in r.failing(True) if r.margins[k][2] == 'validity']}")

## 17. What to replace next

- **Cell data** (`Cell`): use a real datasheet, especially resistance, continuous current and the voltage-versus-charge curve.
- **Motor mass fit** (`motor_mass_lb`) and **motor max voltage**: fit to real products.
- **Propeller efficiency** (`prop_efficiency`): needs a model. Until then pitch and diameter only change rpm, tip speed and Kv.
- **Prismatic coefficient** (`prismatic_coeff`): sets static buoyancy and therefore freeboard.
- **Wave height and current** (`wave_height_in`, `current_mph`): set from the race venue.
- **Placement heights and gyradius**: from the real layout.
- **Deferred hull variables** (transom beam, bow deadrise, keel curve) are not in this notebook; OpenPlaning cannot use them.